<div style="background:linear-gradient(135deg,#2B1B17 0%,#5D4037 55%,#A1887F 100%);color:#FFF8F0;padding:28px 32px;border-radius:14px;">
  <div style="font-size:13px;letter-spacing:3px;text-transform:uppercase;color:#E6C9A8;">Databricks SQL workshop</div>
  <div style="font-size:34px;font-weight:800;margin:6px 0 4px 0;">&#9749; Bean &amp; Brew Coffee Co.</div>
  <div style="font-size:16px;color:#F3E5D8;">Build the demo step by step: clean the raw data, model it, govern it, then create the saved queries, dashboard, Genie agent and alerts.</div>
  <div style="margin-top:16px;display:flex;gap:10px;flex-wrap:wrap;">
    <span style="background:#FFF8F0;color:#3E2723;padding:4px 12px;border-radius:20px;font-size:12px;font-weight:600;">12 stores &middot; 4 regions</span>
    <span style="background:#FFF8F0;color:#3E2723;padding:4px 12px;border-radius:20px;font-size:12px;font-weight:600;">Data: 2025-01-01 &rarr; 2026-10-19</span>
    <span style="background:#FFF8F0;color:#3E2723;padding:4px 12px;border-radius:20px;font-size:12px;font-weight:600;">&ldquo;Today&rdquo; = 2026-10-19</span>
    <span style="background:#FFF8F0;color:#3E2723;padding:4px 12px;border-radius:20px;font-size:12px;font-weight:600;">Schemas: dbsql_workshop_&lt;you&gt;_bronze &middot; _silver &middot; _gold</span>
  </div>
</div>

<div style="background:#FBF6EF;border:1px solid #E6D5C3;border-radius:10px;padding:14px 20px;margin-top:14px;color:#3E2723;">
<b>How to use this notebook.</b> Deploy the <code>dbsql_workshop</code> bundle first (Open in Databricks deploys it for you). The deploy creates a SQL warehouse, your three schemas and the raw bronze tables; this notebook builds everything else. Attach <b>serverless</b> compute (the later steps use Python), then run the cells in order, or click <b>Run all</b>.
<br/><br/>
<b>Safe to re-run.</b> Tables and views are created only if they are missing, and the workspace assets are updated in place. Everything this notebook creates lives in your three schemas or in the bundle's own folder, so the bundle's <b>Destroy</b> removes all of it.
</div>

<div style="background:#FFFFFF;border:1px solid #E6D5C3;border-radius:10px;padding:14px 20px;margin-top:14px;">
<div style="font-size:16px;font-weight:700;color:#5D4037;margin-bottom:6px;">Steps</div>
<ol start="0" style="margin:0;padding-left:20px;line-height:1.8;color:#3E2723;">
<li>Connect to your deployment</li><li>Bronze: look at the raw data</li><li>Silver: clean, conform and deduplicate</li><li>Gold: business-ready tables</li><li>Semantic layer and governance</li><li>Validate</li><li>Deliverables: queries, data prep flow, dashboard, Genie agent, alerts</li><li>Asset index</li>
</ol>
</div>

<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Step 0</div>
<div style="font-size:24px;font-weight:700;">&#128279; Connect to your deployment</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Finds the three schemas the bundle created for you and reads the bundle's settings (SQL warehouse, asset folder, ai_query endpoint, alert subscriber) from the bronze schema's properties. It also sets the session variables <code>bronze_schema</code>, <code>silver_schema</code> and <code>gold_schema</code> that every SQL cell below uses.</p>

In [ ]:
# Connect: find your Bean & Brew schemas and the bundle's settings. Every later step uses them.
import os, sys
sys.path.insert(0, os.path.abspath("src"))
from bean_and_brew import WorkshopAssets, connect, fill

cfg = connect(spark)

<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Step 1</div>
<div style="font-size:24px;font-weight:700;">&#129353; Bronze: loaded by the deploy</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">The deploy already loaded six raw tables (about 1.5M register tickets). The defects are deliberate: mixed-case store IDs, re-sent duplicate tickets, line items as JSON text, upper-case emails. Silver fixes them.</p>

In [ ]:
%sql
-- The six raw tables the deploy loaded
SELECT 'raw_stores' AS table_name, count(*) AS row_count FROM IDENTIFIER(bronze_schema || '.raw_stores')
UNION ALL SELECT 'raw_products', count(*) FROM IDENTIFIER(bronze_schema || '.raw_products')
UNION ALL SELECT 'raw_loyalty_members', count(*) FROM IDENTIFIER(bronze_schema || '.raw_loyalty_members')
UNION ALL SELECT 'raw_pos_transactions', count(*) FROM IDENTIFIER(bronze_schema || '.raw_pos_transactions')
UNION ALL SELECT 'raw_inventory_counts', count(*) FROM IDENTIFIER(bronze_schema || '.raw_inventory_counts')
UNION ALL SELECT 'raw_labor_shifts', count(*) FROM IDENTIFIER(bronze_schema || '.raw_labor_shifts')

In [ ]:
%sql
-- Raw register tickets with a lower-case store ID: note the line items arrive as a JSON string
SELECT * FROM IDENTIFIER(bronze_schema || '.raw_pos_transactions') WHERE store_id <> upper(store_id) LIMIT 5

<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Step 2</div>
<div style="font-size:24px;font-weight:700;">&#129352; Silver: cleaned, governed, related</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Dimensions and facts: types cast, store IDs upper-cased, duplicate re-sends removed, JSON line items exploded.</p>


In [ ]:
%sql
-- Silver dimensions: typed, trimmed, IDs and emails normalized
CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.dim_store')
COMMENT 'Conformed store dimension for Bean & Brew Coffee Co. One row per store location with typed attributes. Source: bronze.raw_stores.'
AS SELECT upper(trim(store_id)) AS store_id, store_name, city, state, region, store_format,
  to_date(open_date) AS open_date, cast(square_feet AS INT) AS square_feet, cast(seating_capacity AS INT) AS seating_capacity,
  store_manager, cast(latitude AS DOUBLE) latitude, cast(longitude AS DOUBLE) longitude
FROM IDENTIFIER(bronze_schema || '.raw_stores');

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.dim_product')
COMMENT 'Conformed product/menu dimension. One row per sellable SKU with 2025 list price, standard unit cost, and standard margin. Source: bronze.raw_products.'
AS SELECT product_id, product_name, category, temperature AS serving_type,
  cast(list_price AS DECIMAL(10,2)) AS list_price_2025,
  cast(round(list_price * 1.05, 2) AS DECIMAL(10,2)) AS list_price_2026,
  cast(unit_cost AS DECIMAL(10,2)) AS unit_cost,
  cast(round((list_price - unit_cost) / list_price, 4) AS DECIMAL(6,4)) AS standard_margin_pct,
  cast(is_seasonal AS BOOLEAN) AS is_seasonal,
  category IN ('Espresso Drinks','Brewed Coffee','Cold Drinks','Tea','Seasonal') AS is_beverage
FROM IDENTIFIER(bronze_schema || '.raw_products');

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.dim_loyalty_member')
COMMENT 'Cleaned Bean & Brew Rewards loyalty member dimension. Emails lower-cased, home store IDs normalized, dates typed. Source: bronze.raw_loyalty_members.'
AS SELECT member_id, first_name, last_name, lower(trim(email)) AS email, to_date(signup_date) AS signup_date,
  upper(home_store_id) AS home_store_id, tier AS loyalty_tier, birth_month
FROM IDENTIFIER(bronze_schema || '.raw_loyalty_members');

In [ ]:
%sql
-- Silver facts: duplicates removed, voids excluded from line items, JSON exploded, price book and promos applied
CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.fact_sales_line_items')
COMMENT 'Cleaned POS line-item fact. One row per completed ticket line. Duplicate POS re-sends removed, voided tickets excluded, JSON payload exploded, prices resolved from the active price book, promo discounts applied. Sources: bronze.raw_pos_transactions, silver.dim_product.'
AS
WITH dedup AS (
  SELECT * EXCEPT (rn) FROM (
    SELECT *, row_number() OVER (PARTITION BY transaction_id ORDER BY _ingested_at DESC) rn FROM IDENTIFIER(bronze_schema || '.raw_pos_transactions')) WHERE rn = 1),
exploded AS (
  SELECT d.transaction_id, upper(d.store_id) AS store_id, to_timestamp(d.transaction_ts) AS transaction_ts,
    d.order_channel, d.loyalty_member_id, d.promo_code, d.price_book, li.line, li.sku, li.qty
  FROM dedup d
  LATERAL VIEW explode(from_json(d.line_items_json, 'array<struct<line:int,sku:string,qty:int>>')) e AS li
  WHERE d.ticket_status = 'COMPLETED'),
priced AS (
  SELECT
    concat(e.transaction_id, '-', e.line) AS line_item_id,
    e.transaction_id, e.line AS line_number, e.store_id, e.transaction_ts,
    to_date(e.transaction_ts) AS business_date, hour(e.transaction_ts) AS business_hour,
    e.order_channel, e.loyalty_member_id, e.promo_code,
    e.sku AS product_id, e.qty AS quantity,
    CASE WHEN e.price_book = 'menu_v2026' THEN p.list_price_2026 ELSE p.list_price_2025 END AS unit_price,
    cast(e.qty * CASE WHEN e.price_book = 'menu_v2026' THEN p.list_price_2026 ELSE p.list_price_2025 END AS DECIMAL(12,2)) AS gross_sales,
    cast(round(e.qty * CASE WHEN e.price_book = 'menu_v2026' THEN p.list_price_2026 ELSE p.list_price_2025 END *
      CASE WHEN e.promo_code = 'LOYALTY10' THEN 0.10 WHEN e.promo_code = 'HAPPYHOUR' AND p.is_beverage THEN 0.20 ELSE 0 END, 2) AS DECIMAL(12,2)) AS discount_amount,
    cast(e.qty * p.unit_cost AS DECIMAL(12,2)) AS cost_of_goods
  FROM exploded e JOIN IDENTIFIER(silver_schema || '.dim_product') p ON e.sku = p.product_id)
SELECT *, cast(gross_sales - discount_amount AS DECIMAL(12,2)) AS net_sales FROM priced;

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.fact_transactions')
COMMENT 'Cleaned POS ticket (transaction header) fact. One row per unique ticket, including voided tickets (flagged). Totals rolled up from line items; sales tax applied by store state. Sources: bronze.raw_pos_transactions, silver.fact_sales_line_items, silver.dim_store.'
AS
WITH dedup AS (
  SELECT * EXCEPT (rn) FROM (
    SELECT *, row_number() OVER (PARTITION BY transaction_id ORDER BY _ingested_at DESC) rn FROM IDENTIFIER(bronze_schema || '.raw_pos_transactions')) WHERE rn = 1),
items AS (
  SELECT transaction_id, count(*) line_count, sum(quantity) item_qty, sum(gross_sales) gross_sales, sum(discount_amount) discount_amount,
    sum(net_sales) net_sales, sum(cost_of_goods) cost_of_goods
  FROM IDENTIFIER(silver_schema || '.fact_sales_line_items') GROUP BY transaction_id)
SELECT d.transaction_id, upper(d.store_id) AS store_id, d.register_id, to_timestamp(d.transaction_ts) AS transaction_ts,
  to_date(to_timestamp(d.transaction_ts)) AS business_date, hour(to_timestamp(d.transaction_ts)) AS business_hour,
  d.order_channel, d.loyalty_member_id, d.loyalty_member_id IS NOT NULL AS is_loyalty_transaction,
  d.payment_method, d.promo_code, d.ticket_status = 'VOIDED' AS is_voided,
  coalesce(i.line_count,0) line_count, coalesce(i.item_qty,0) item_qty,
  coalesce(i.gross_sales,0) gross_sales, coalesce(i.discount_amount,0) discount_amount, coalesce(i.net_sales,0) net_sales,
  cast(round(coalesce(i.net_sales,0) * CASE s.state WHEN 'WA' THEN 0.1025 WHEN 'OR' THEN 0 WHEN 'CA' THEN 0.0875 WHEN 'AZ' THEN 0.086 WHEN 'TX' THEN 0.0825 ELSE 0.0881 END, 2) AS DECIMAL(12,2)) AS sales_tax,
  coalesce(i.cost_of_goods,0) cost_of_goods
FROM dedup d LEFT JOIN items i USING (transaction_id) JOIN IDENTIFIER(silver_schema || '.dim_store') s ON upper(d.store_id) = s.store_id;

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.fact_inventory_daily')
COMMENT 'Cleaned end-of-day inventory position per store and stock item, with valuation and reorder flag. Source: bronze.raw_inventory_counts.'
AS SELECT upper(store_id) store_id, count_date, item_code, item_name, unit_of_measure,
  cast(on_hand_qty AS DECIMAL(12,1)) on_hand_qty, cast(par_level AS DECIMAL(12,1)) par_level, cast(reorder_point AS DECIMAL(12,1)) reorder_point,
  cast(unit_cost AS DECIMAL(10,2)) unit_cost, cast(on_hand_qty * unit_cost AS DECIMAL(12,2)) AS inventory_value,
  on_hand_qty < reorder_point AS is_below_reorder_point
FROM IDENTIFIER(bronze_schema || '.raw_inventory_counts');

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.fact_labor_daily')
COMMENT 'Cleaned daily labor fact per store from the WFM system, with computed labor cost. Source: bronze.raw_labor_shifts.'
AS SELECT upper(store_id) store_id, business_date, cast(scheduled_hours AS DECIMAL(8,1)) scheduled_hours, cast(worked_hours AS DECIMAL(8,1)) worked_hours,
  headcount, cast(avg_hourly_wage AS DECIMAL(8,2)) avg_hourly_wage, cast(worked_hours * avg_hourly_wage AS DECIMAL(12,2)) AS labor_cost
FROM IDENTIFIER(bronze_schema || '.raw_labor_shifts');

<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Step 3</div>
<div style="font-size:24px;font-weight:700;">&#129351; Gold: business-ready tables</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Store, product, hourly, inventory and customer tables at the grain each business question needs, plus the promo target list.</p>


In [ ]:
%sql
-- Gold: one table per business grain
CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.store_daily_performance')
COMMENT 'Gold: daily store P&L-lite. One row per store per business date with traffic, sales, margin, loyalty mix, and labor. Excludes voided tickets. Primary table for store performance dashboards, Genie, and alerts. Sources: silver.fact_transactions, silver.fact_labor_daily, silver.dim_store.'
AS
WITH t AS (
  SELECT store_id, business_date,
    count(*) transactions, sum(item_qty) items_sold, sum(gross_sales) gross_sales, sum(discount_amount) discount_amount,
    sum(net_sales) net_sales, sum(cost_of_goods) cost_of_goods,
    count_if(is_loyalty_transaction) loyalty_transactions,
    count_if(order_channel IN ('Mobile App','Delivery')) digital_transactions,
    count_if(order_channel = 'Drive-Thru') drive_thru_transactions
  FROM IDENTIFIER(silver_schema || '.fact_transactions') WHERE NOT is_voided GROUP BY ALL)
SELECT t.business_date, t.store_id, s.store_name, s.city, s.state, s.region, s.store_format,
  datediff(t.business_date, s.open_date) >= 365 AS is_comp_store,
  t.transactions, t.items_sold, t.gross_sales, t.discount_amount, t.net_sales, t.cost_of_goods,
  t.net_sales - t.cost_of_goods AS gross_profit,
  cast(round(t.net_sales / t.transactions, 2) AS DECIMAL(10,2)) AS avg_ticket,
  t.loyalty_transactions, t.digital_transactions, t.drive_thru_transactions,
  l.worked_hours AS labor_hours, l.labor_cost,
  cast(round(l.labor_cost / t.net_sales, 4) AS DECIMAL(8,4)) AS labor_cost_pct
FROM t JOIN IDENTIFIER(silver_schema || '.dim_store') s USING (store_id)
LEFT JOIN IDENTIFIER(silver_schema || '.fact_labor_daily') l ON l.store_id = t.store_id AND l.business_date = t.business_date;

CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.product_sales_daily')
COMMENT 'Gold: daily product sales by store. One row per business date, store, and product with units, sales, and margin. Sources: silver.fact_sales_line_items, silver.dim_product, silver.dim_store.'
AS SELECT f.business_date, f.store_id, s.store_name, s.region, f.product_id, p.product_name, p.category, p.serving_type, p.is_beverage,
  sum(f.quantity) units_sold, sum(f.gross_sales) gross_sales, sum(f.discount_amount) discount_amount, sum(f.net_sales) net_sales,
  sum(f.cost_of_goods) cost_of_goods, sum(f.net_sales) - sum(f.cost_of_goods) gross_profit
FROM IDENTIFIER(silver_schema || '.fact_sales_line_items') f JOIN IDENTIFIER(silver_schema || '.dim_product') p USING (product_id) JOIN IDENTIFIER(silver_schema || '.dim_store') s USING (store_id)
GROUP BY ALL;

CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.store_hourly_traffic')
COMMENT 'Gold: hourly ticket counts and sales per store and day, for staffing and peak-hour analysis. Excludes voided tickets. Source: silver.fact_transactions.'
AS SELECT store_id, business_date, business_hour, date_format(business_date,'EEEE') AS day_of_week,
  count(*) transactions, sum(net_sales) net_sales, count_if(order_channel = 'Mobile App') mobile_transactions
FROM IDENTIFIER(silver_schema || '.fact_transactions') WHERE NOT is_voided GROUP BY ALL;

CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.inventory_current_status')
COMMENT 'Gold: latest inventory position per store and stock item with stock health status (OK / LOW / REORDER NOW). Source: silver.fact_inventory_daily, silver.dim_store.'
AS
WITH latest AS (SELECT * FROM IDENTIFIER(silver_schema || '.fact_inventory_daily') QUALIFY row_number() OVER (PARTITION BY store_id, item_code ORDER BY count_date DESC) = 1)
SELECT l.store_id, s.store_name, s.region, l.count_date AS as_of_date, l.item_code, l.item_name, l.unit_of_measure,
  l.on_hand_qty, l.par_level, l.reorder_point, cast(round(l.on_hand_qty / l.par_level, 3) AS DECIMAL(6,3)) AS pct_of_par,
  l.inventory_value,
  CASE WHEN l.on_hand_qty < l.reorder_point THEN 'REORDER NOW' WHEN l.on_hand_qty < l.reorder_point * 1.5 THEN 'LOW' ELSE 'OK' END AS stock_status
FROM latest l JOIN IDENTIFIER(silver_schema || '.dim_store') s USING (store_id);

-- mode(x, true) breaks ties on the lowest value, so favorite store/product are stable across runs
CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.customer_360')
COMMENT 'Gold: one row per Bean & Brew Rewards member with lifetime value, visit frequency, recency, favorite store/product, and value segment (Champion = top 10% by lifetime spend, Loyal Regular = next 30%, Occasional = next 40%, Light = bottom 20%). Sources: silver.dim_loyalty_member, silver.fact_transactions, silver.fact_sales_line_items, silver.dim_product.'
AS
WITH tx AS (
  SELECT loyalty_member_id member_id, count(*) visits, sum(net_sales) lifetime_net_sales, min(business_date) first_visit_date, max(business_date) last_visit_date,
    mode(store_id, true) favorite_store_id, count_if(order_channel = 'Mobile App') mobile_visits
  FROM IDENTIFIER(silver_schema || '.fact_transactions') WHERE NOT is_voided AND loyalty_member_id IS NOT NULL GROUP BY 1),
fav AS (
  SELECT loyalty_member_id member_id, mode(p.product_name, true) favorite_product
  FROM IDENTIFIER(silver_schema || '.fact_sales_line_items') f JOIN IDENTIFIER(silver_schema || '.dim_product') p USING (product_id) WHERE loyalty_member_id IS NOT NULL GROUP BY 1),
base AS (
  SELECT m.member_id, m.first_name, m.last_name, m.loyalty_tier, m.signup_date, m.home_store_id,
    tx.visits, tx.lifetime_net_sales, cast(round(tx.lifetime_net_sales / tx.visits, 2) AS DECIMAL(10,2)) avg_ticket,
    tx.first_visit_date, tx.last_visit_date, datediff(DATE'2026-10-19', tx.last_visit_date) AS days_since_last_visit,
    tx.favorite_store_id, fav.favorite_product, round(tx.mobile_visits / tx.visits, 3) AS mobile_share,
    percent_rank() OVER (ORDER BY tx.lifetime_net_sales DESC, m.member_id) pr
  FROM IDENTIFIER(silver_schema || '.dim_loyalty_member') m LEFT JOIN tx USING (member_id) LEFT JOIN fav USING (member_id))
SELECT * EXCEPT (pr),
  CASE WHEN pr < 0.10 THEN 'Champion' WHEN pr < 0.40 THEN 'Loyal Regular' WHEN pr < 0.80 THEN 'Occasional' ELSE 'Light' END AS customer_segment
FROM base;

In [ ]:
%sql
-- Same logic as the "Loyalty Promo Target List Prep" visual data prep flow (deployed below), so the
-- table exists before the demo. Running the flow live overwrites it with identical rows.
-- CHAR(36) is '$': a literal dollar-digit is read as a parameter in notebook SQL.
CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.loyalty_promo_targets') AS
SELECT c.member_id, c.first_name, c.loyalty_tier, c.customer_segment, c.lifetime_net_sales, c.visits, c.favorite_product,
       s.store_name, s.region,
       CASE WHEN c.favorite_product LIKE '%Cold Brew%' THEN 'Free Nitro Upgrade'
            WHEN c.favorite_product LIKE '%Latte%'     THEN 'Free Oat Milk Swap'
            ELSE CONCAT(CHAR(36), '2 Off Pastry') END AS promo_offer
FROM IDENTIFIER(gold_schema || '.customer_360') c
JOIN IDENTIFIER(silver_schema || '.dim_store') s ON c.home_store_id = s.store_id
WHERE c.customer_segment IN ('Champion', 'Loyal Regular') AND c.loyalty_tier IN ('Gold', 'Reserve');

<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Step 4</div>
<div style="font-size:24px;font-weight:700;">&#128207; Semantic layer and governance</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Three Unity Catalog metric views (one definition of every KPI), primary and foreign keys, column descriptions and governed tags.</p>


In [ ]:
# Metric views: the YAML is the original definition, pointed at the bundle's catalog and schemas
METRIC_VIEWS = {
    'store_sales_metrics': r'''CREATE VIEW IF NOT EXISTS __CATALOG__.__GOLD__.store_sales_metrics
WITH METRICS LANGUAGE YAML
COMMENT 'Certified Bean & Brew store performance KPIs: net sales, traffic, average ticket, margin, labor and loyalty mix. Use for any store-level or regional sales question.'
AS $$
version: 1.1
source: __CATALOG__.__GOLD__.store_daily_performance
comment: Certified Bean & Brew store performance KPIs. One source row = one store-day. Excludes voided tickets.
dimensions:
  - name: business_date
    expr: business_date
    display_name: Business Date
    comment: Store-local business date.
    format:
      type: date
      date_format: year_month_day
    synonyms: [date, day, sales date]
  - name: business_month
    expr: DATE_TRUNC('MONTH', business_date)
    display_name: Month
    comment: Calendar month of the business date.
    format:
      type: date
      date_format: locale_short_month
    synonyms: [month, sales month]
  - name: business_week
    expr: DATE_TRUNC('WEEK', business_date)
    display_name: Week
    comment: Week (Monday start) of the business date.
    synonyms: [week]
  - name: day_of_week
    expr: date_format(business_date, 'EEEE')
    display_name: Day of Week
    synonyms: [weekday]
  - name: store_name
    expr: store_name
    display_name: Store
    comment: Customer-facing store name.
    synonyms: [store, location, cafe, shop]
  - name: store_id
    expr: store_id
    display_name: Store ID
  - name: city
    expr: city
    display_name: City
  - name: region
    expr: region
    display_name: Region
    comment: Operating region (Pacific Northwest, California, Southwest, Mountain).
    synonyms: [market, territory]
  - name: store_format
    expr: store_format
    display_name: Store Format
    comment: Flagship Cafe, Cafe, Drive-Thru, or Express Kiosk.
    synonyms: [format, store type]
  - name: is_comp_store
    expr: is_comp_store
    display_name: Comp Store
    comment: True when the store has been open 365+ days (comparable-store sales).
    synonyms: [comparable store, same store]
measures:
  - name: net_sales
    expr: SUM(net_sales)
    display_name: Net Sales
    comment: Gross sales minus discounts, pre-tax. Primary revenue KPI.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [revenue, sales, total sales, turnover]
  - name: transactions
    expr: SUM(transactions)
    display_name: Transactions
    comment: Completed tickets, used as a proxy for customer traffic.
    format:
      type: number
      decimal_places: {type: exact, places: 0}
    synonyms: [tickets, orders, traffic, visits, customer count]
  - name: avg_ticket
    expr: SUM(net_sales) / SUM(transactions)
    display_name: Average Ticket
    comment: Net sales divided by transactions.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 2}
    synonyms: [average order value, AOV, average check, basket size]
  - name: gross_profit
    expr: SUM(gross_profit)
    display_name: Gross Profit
    comment: Net sales minus cost of goods sold.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [margin dollars, profit]
  - name: gross_margin_pct
    expr: SUM(gross_profit) / SUM(net_sales)
    display_name: Gross Margin %
    comment: Gross profit as a share of net sales.
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [margin, GM%]
  - name: labor_cost_pct
    expr: SUM(labor_cost) / SUM(net_sales)
    display_name: Labor Cost %
    comment: Labor cost as a share of net sales. Target is below 30%.
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [labor percent, labor ratio, wage ratio]
  - name: loyalty_mix_pct
    expr: SUM(loyalty_transactions) / SUM(transactions)
    display_name: Loyalty Mix %
    comment: Share of tickets with a Bean & Brew Rewards member attached.
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [rewards penetration, loyalty share]
  - name: digital_mix_pct
    expr: SUM(digital_transactions) / SUM(transactions)
    display_name: Digital Mix %
    comment: Share of tickets ordered via Mobile App or Delivery.
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [mobile share, digital share, app orders share]
  - name: discount_rate
    expr: SUM(discount_amount) / SUM(gross_sales)
    display_name: Discount Rate
    comment: Promotional discounts as a share of gross sales.
    format:
      type: percentage
      decimal_places: {type: exact, places: 2}
    synonyms: [promo rate, markdown rate]
  - name: store_days
    expr: COUNT(1)
    display_name: Store Operating Days
    comment: Number of store-days with sales.
$$''',
    'product_sales_metrics': r'''CREATE VIEW IF NOT EXISTS __CATALOG__.__GOLD__.product_sales_metrics
WITH METRICS LANGUAGE YAML
COMMENT 'Certified Bean & Brew menu/product KPIs at line-item grain with star-schema joins to product and store dimensions. Use for product mix, category, daypart, and channel questions.'
AS $$
version: 1.1
source: __CATALOG__.__SILVER__.fact_sales_line_items
comment: Product mix KPIs. Grain = one POS line item. Joins product and store dimensions (many-to-one, PK-enforced).
joins:
  - name: product
    source: __CATALOG__.__SILVER__.dim_product
    on: source.product_id = product.product_id
    cardinality: many_to_one
    rely:
      at_most_one_match: true
  - name: store
    source: __CATALOG__.__SILVER__.dim_store
    on: source.store_id = store.store_id
    cardinality: many_to_one
    rely:
      at_most_one_match: true
dimensions:
  - name: business_date
    expr: source.business_date
    display_name: Business Date
    synonyms: [date, day]
  - name: business_month
    expr: DATE_TRUNC('MONTH', source.business_date)
    display_name: Month
    synonyms: [month]
  - name: business_hour
    expr: source.business_hour
    display_name: Hour of Day
    comment: Hour (0-23) the ticket was rung up, store-local.
    synonyms: [hour, time of day]
  - name: daypart
    expr: |-
      CASE WHEN source.business_hour < 11 THEN 'Morning Rush (6-11)'
           WHEN source.business_hour < 14 THEN 'Midday (11-14)'
           WHEN source.business_hour < 17 THEN 'Afternoon (14-17)'
           ELSE 'Evening (17+)' END
    display_name: Daypart
    comment: Operational daypart bucket.
    synonyms: [shift, time of day bucket]
  - name: order_channel
    expr: source.order_channel
    display_name: Order Channel
    comment: In-Store, Mobile App, Drive-Thru, or Delivery.
    synonyms: [channel, ordering method]
  - name: product_name
    expr: product.product_name
    display_name: Product
    synonyms: [item, drink, menu item, sku]
  - name: category
    expr: product.category
    display_name: Category
    comment: Menu category (Espresso Drinks, Brewed Coffee, Cold Drinks, Tea, Bakery, Breakfast, Seasonal, Retail Beans).
    synonyms: [product category, menu category]
  - name: serving_type
    expr: product.serving_type
    display_name: Serving Type
    comment: Hot, Iced, Food, or Retail.
    synonyms: [hot or iced, temperature]
  - name: is_beverage
    expr: product.is_beverage
    display_name: Is Beverage
  - name: store_name
    expr: store.store_name
    display_name: Store
    synonyms: [store, location, cafe]
  - name: region
    expr: store.region
    display_name: Region
    synonyms: [market]
  - name: is_loyalty_member
    expr: source.loyalty_member_id IS NOT NULL
    display_name: Loyalty Member Purchase
    synonyms: [rewards member, member purchase]
measures:
  - name: units_sold
    expr: SUM(source.quantity)
    display_name: Units Sold
    comment: Total item quantity sold.
    format:
      type: number
      decimal_places: {type: exact, places: 0}
    synonyms: [units, quantity, items sold, cups sold]
  - name: net_sales
    expr: SUM(source.net_sales)
    display_name: Net Sales
    comment: Gross sales minus discounts, pre-tax.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [revenue, sales]
  - name: gross_profit
    expr: SUM(source.net_sales) - SUM(source.cost_of_goods)
    display_name: Gross Profit
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [profit, margin dollars]
  - name: gross_margin_pct
    expr: (SUM(source.net_sales) - SUM(source.cost_of_goods)) / SUM(source.net_sales)
    display_name: Gross Margin %
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [margin, GM%]
  - name: discount_amount
    expr: SUM(source.discount_amount)
    display_name: Discounts
    comment: Promotional discounts given (LOYALTY10, HAPPYHOUR).
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
    synonyms: [promo cost, markdowns]
  - name: tickets
    expr: COUNT(DISTINCT source.transaction_id)
    display_name: Tickets Containing Item
    comment: Distinct tickets that included the item(s) in the current grouping.
    synonyms: [orders, attach count]
$$''',
    'customer_metrics': r'''CREATE VIEW IF NOT EXISTS __CATALOG__.__GOLD__.customer_metrics
WITH METRICS LANGUAGE YAML
COMMENT 'Certified Bean & Brew Rewards loyalty KPIs: member counts, lifetime value, visit frequency, and mobile adoption by tier and segment.'
AS $$
version: 1.1
source: __CATALOG__.__GOLD__.customer_360
comment: Loyalty member KPIs. Grain = one Bean & Brew Rewards member.
dimensions:
  - name: loyalty_tier
    expr: loyalty_tier
    display_name: Loyalty Tier
    comment: Green (entry), Gold, Reserve (top tier).
    synonyms: [tier, rewards tier, membership level]
  - name: customer_segment
    expr: customer_segment
    display_name: Customer Segment
    comment: Champion (top 10% spend), Loyal Regular, Occasional, Light (bottom 20%).
    synonyms: [segment, value segment, RFM segment]
  - name: favorite_product
    expr: favorite_product
    display_name: Favorite Product
    synonyms: [usual order, go-to drink]
  - name: favorite_store_id
    expr: favorite_store_id
    display_name: Favorite Store ID
  - name: signup_year
    expr: year(signup_date)
    display_name: Signup Year
    synonyms: [cohort, enrollment year]
measures:
  - name: member_count
    expr: COUNT(1)
    display_name: Members
    synonyms: [members, customers, number of members]
  - name: total_lifetime_value
    expr: SUM(lifetime_net_sales)
    display_name: Total Lifetime Value
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [member revenue, total LTV]
  - name: avg_lifetime_value
    expr: AVG(lifetime_net_sales)
    display_name: Avg Lifetime Value
    comment: Average net sales per member since 2025-01-01.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 2}
    synonyms: [LTV, CLV, average customer value]
  - name: avg_visits
    expr: AVG(visits)
    display_name: Avg Visits per Member
    format:
      type: number
      decimal_places: {type: exact, places: 1}
    synonyms: [visit frequency, frequency]
  - name: avg_mobile_share
    expr: AVG(mobile_share)
    display_name: Avg Mobile Order Share
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [app adoption, mobile adoption]
$$''',
}

for name, ddl in METRIC_VIEWS.items():
    spark.sql(fill(ddl, cfg))
    print(f"Created metric view {cfg['gold_schema']}.{name}")

In [ ]:
# Keys, column descriptions and governed tags
CONSTRAINTS = [
 [
  "__SILVER__.dim_store",
  "pk_dim_store",
  "PRIMARY KEY (store_id)"
 ],
 [
  "__SILVER__.dim_product",
  "pk_dim_product",
  "PRIMARY KEY (product_id)"
 ],
 [
  "__SILVER__.dim_loyalty_member",
  "pk_dim_loyalty_member",
  "PRIMARY KEY (member_id)"
 ],
 [
  "__SILVER__.fact_transactions",
  "pk_fact_transactions",
  "PRIMARY KEY (transaction_id)"
 ],
 [
  "__SILVER__.fact_sales_line_items",
  "pk_fact_sales_line_items",
  "PRIMARY KEY (line_item_id)"
 ],
 [
  "__GOLD__.customer_360",
  "pk_customer_360",
  "PRIMARY KEY (member_id)"
 ],
 [
  "__SILVER__.dim_loyalty_member",
  "fk_member_home_store",
  "FOREIGN KEY (home_store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ],
 [
  "__SILVER__.fact_transactions",
  "fk_txn_store",
  "FOREIGN KEY (store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ],
 [
  "__SILVER__.fact_transactions",
  "fk_txn_member",
  "FOREIGN KEY (loyalty_member_id) REFERENCES __SILVER__.dim_loyalty_member(member_id)"
 ],
 [
  "__SILVER__.fact_sales_line_items",
  "fk_line_txn",
  "FOREIGN KEY (transaction_id) REFERENCES __SILVER__.fact_transactions(transaction_id)"
 ],
 [
  "__SILVER__.fact_sales_line_items",
  "fk_line_product",
  "FOREIGN KEY (product_id) REFERENCES __SILVER__.dim_product(product_id)"
 ],
 [
  "__SILVER__.fact_sales_line_items",
  "fk_line_store",
  "FOREIGN KEY (store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ],
 [
  "__SILVER__.fact_inventory_daily",
  "fk_inv_store",
  "FOREIGN KEY (store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ],
 [
  "__SILVER__.fact_labor_daily",
  "fk_labor_store",
  "FOREIGN KEY (store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ]
]

COLUMN_COMMENTS = {
 "__GOLD__.customer_360": {
  "loyalty_tier": "Rewards tier: Green (entry), Gold, Reserve (top tier).",
  "home_store_id": "Store the member enrolled at. FK to dim_store.",
  "lifetime_net_sales": "Total net sales (pre-tax, after discounts) across all non-voided visits since 2025-01-01, USD.",
  "days_since_last_visit": "Days between last visit and 2026-10-19 (data as-of date).",
  "customer_segment": "Value segment by lifetime spend percentile: Champion (top 10%), Loyal Regular (next 30%), Occasional (next 40%), Light (bottom 20%)."
 },
 "__GOLD__.inventory_current_status": {
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain.",
  "stock_status": "REORDER NOW = below reorder point; LOW = below 1.5x reorder point; OK otherwise."
 },
 "__GOLD__.loyalty_promo_targets": {
  "loyalty_tier": "Rewards tier: Green (entry), Gold, Reserve (top tier).",
  "customer_segment": "Value segment by lifetime spend percentile: Champion (top 10%), Loyal Regular (next 30%), Occasional (next 40%), Light (bottom 20%).",
  "lifetime_net_sales": "Total net sales (pre-tax, after discounts) across all non-voided visits since 2025-01-01, USD.",
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain."
 },
 "__GOLD__.product_sales_daily": {
  "business_date": "Store-local business date of the sale.",
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain.",
  "category": "Menu category: Espresso Drinks, Brewed Coffee, Cold Drinks, Tea, Bakery, Breakfast, Seasonal, Retail Beans.",
  "is_beverage": "True for drink categories; false for food and retail beans."
 },
 "__GOLD__.store_daily_performance": {
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain.",
  "store_format": "Store format: Flagship Cafe, Cafe, Drive-Thru, or Express Kiosk.",
  "is_comp_store": "True when the store has been open at least 365 days as of business_date (comparable-store sales).",
  "transactions": "Count of completed (non-voided) tickets. Proxy for customer traffic.",
  "net_sales": "Net sales = gross sales - discounts, pre-tax, USD. Primary revenue measure.",
  "gross_profit": "net_sales - cost_of_goods, USD.",
  "avg_ticket": "Average ticket = net_sales / transactions, USD.",
  "digital_transactions": "Tickets ordered via Mobile App or Delivery.",
  "labor_cost": "worked_hours * avg_hourly_wage, USD (excludes benefits).",
  "labor_cost_pct": "labor_cost / net_sales. Target is below 0.30."
 },
 "__SILVER__.dim_loyalty_member": {
  "email": "Member email address (PII). Lower-cased.",
  "home_store_id": "Store the member enrolled at. FK to dim_store.",
  "loyalty_tier": "Rewards tier: Green (entry), Gold, Reserve (top tier)."
 },
 "__SILVER__.dim_product": {
  "category": "Menu category: Espresso Drinks, Brewed Coffee, Cold Drinks, Tea, Bakery, Breakfast, Seasonal, Retail Beans.",
  "list_price_2025": "Menu list price in USD under the 2025 price book.",
  "list_price_2026": "Menu list price in USD under the 2026 price book (+5%, effective 2026-01-05).",
  "unit_cost": "Standard cost of goods per unit in USD (ingredients + packaging).",
  "standard_margin_pct": "Standard gross margin at 2025 list price: (list_price - unit_cost) / list_price.",
  "is_beverage": "True for drink categories; false for food and retail beans."
 },
 "__SILVER__.dim_store": {
  "store_id": "Unique store identifier (e.g. S001). Primary key.",
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain.",
  "store_format": "Store format: Flagship Cafe, Cafe, Drive-Thru, or Express Kiosk.",
  "open_date": "Date the store opened to the public. Stores open < 12 months are considered new (non-comp) stores.",
  "seating_capacity": "Number of customer seats."
 },
 "__SILVER__.fact_inventory_daily": {
  "is_below_reorder_point": "True when on-hand quantity is below the reorder point and a replenishment order is required."
 },
 "__SILVER__.fact_labor_daily": {
  "labor_cost": "worked_hours * avg_hourly_wage, USD (excludes benefits)."
 },
 "__SILVER__.fact_sales_line_items": {
  "business_date": "Store-local business date of the sale.",
  "business_hour": "Hour of day (0-23) the ticket was rung up, store-local.",
  "order_channel": "How the order was placed: In-Store, Mobile App, Drive-Thru, Delivery.",
  "gross_sales": "quantity * unit_price before discounts, USD.",
  "discount_amount": "Promotional discount: LOYALTY10 = 10% off; HAPPYHOUR = 20% off beverages.",
  "cost_of_goods": "quantity * standard unit cost, USD.",
  "net_sales": "gross_sales - discount_amount, pre-tax."
 },
 "__SILVER__.fact_transactions": {
  "is_loyalty_transaction": "True if a Bean & Brew Rewards member was attached to the ticket.",
  "is_voided": "True if the ticket was voided at the register; excluded from sales metrics.",
  "net_sales": "Ticket net sales (gross - discounts), pre-tax, USD. Zero for voided tickets.",
  "sales_tax": "Sales tax collected based on store state rate."
 }
}

TAGS = [
 "ALTER SCHEMA __BRONZE__ SET TAGS ('layer' = 'bronze', 'domain' = 'sales')",
 "ALTER SCHEMA __SILVER__ SET TAGS ('layer' = 'silver', 'domain' = 'sales')",
 "ALTER SCHEMA __GOLD__ SET TAGS ('layer' = 'gold', 'domain' = 'sales')",
 "ALTER TABLE __SILVER__.fact_inventory_daily SET TAGS ('domain' = 'supply_chain')",
 "ALTER TABLE __GOLD__.inventory_current_status SET TAGS ('domain' = 'supply_chain')",
 "ALTER TABLE __SILVER__.fact_labor_daily SET TAGS ('domain' = 'hr')",
 "ALTER TABLE __SILVER__.dim_loyalty_member ALTER COLUMN email SET TAGS ('contains_pii' = 'true')",
 "ALTER TABLE __BRONZE__.raw_loyalty_members ALTER COLUMN email SET TAGS ('contains_pii' = 'true')",
 "ALTER TABLE __GOLD__.store_daily_performance SET TAGS ('owner_team' = 'analytics')",
 "ALTER TABLE __GOLD__.customer_360 SET TAGS ('owner_team' = 'customer_data')"
]

# Point the definitions above at the bundle's schemas
CONSTRAINTS = [[fill(t, cfg), n, fill(ddl, cfg)] for t, n, ddl in CONSTRAINTS]
COLUMN_COMMENTS = {fill(t, cfg): cols for t, cols in COLUMN_COMMENTS.items()}
TAGS = [fill(s, cfg) for s in TAGS]

existing = {(f"{r.table_schema}.{r.table_name}", r.constraint_name) for r in spark.sql(fill(
    "SELECT table_schema, table_name, constraint_name FROM information_schema.table_constraints WHERE table_schema IN ('__BRONZE__', '__SILVER__', '__GOLD__')", cfg)).collect()}
added = 0
for table, name, ddl in CONSTRAINTS:  # primary keys first, then the foreign keys that reference them
    if (table, name) in existing:
        continue
    added += 1
    if ddl.startswith("PRIMARY KEY"):  # key columns must be NOT NULL; CTAS doesn't always infer that
        for col in ddl[ddl.index("(") + 1:ddl.index(")")].split(","):
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {col.strip()} SET NOT NULL")
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} {ddl}")
print(f"Added {added} primary and foreign keys ({len(CONSTRAINTS) - added} already existed)")

n = 0
for table, cols in COLUMN_COMMENTS.items():
    for col, text in cols.items():
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN `{col}` COMMENT '{text.replace(chr(39), chr(92) + chr(39))}'")
        n += 1
print(f"Set {n} column comments")

# Tags may be governed by account-level tag policies with restricted values, so a failed tag is a warning
failed = []
for stmt in TAGS:
    try:
        spark.sql(stmt)
    except Exception as e:
        failed.append((stmt, str(e).splitlines()[0][:160]))
print(f"Applied {len(TAGS) - len(failed)} of {len(TAGS)} tags")
for stmt, err in failed:
    print(f"  WARNING tag not applied: {stmt}\n    {err}")

<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Step 5</div>
<div style="font-size:24px;font-weight:700;">&#9989; Validate the build</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Checks every demo scenario the demo relies on and prints a data fingerprint that is identical on every rebuild. A failed check stops the notebook here, before any workspace asset is created.</p>

In [ ]:
# Every query below names the bundle's schemas through placeholders filled in by sql()
def sql(q):
    return spark.sql(fill(q, cfg))

EXPECTED_FINGERPRINT = 'ab02b95e172468b55ac67d170273121b11f8e795348cd90d4f8ac547a17ba1bc'

# Validation: every scenario the demo relies on, plus a fingerprint that is identical on every rebuild
def one(q):
    return sql(q).first()

checks = []
def check(name, ok, detail):
    checks.append((("PASS" if ok else "FAIL"), name, str(detail)))

b = one("SELECT count(*) n, count(*) - count(DISTINCT transaction_id) dups, count_if(store_id <> upper(store_id)) lower FROM __BRONZE__.raw_pos_transactions")
s = one("SELECT count(*) n, count(*) - count(DISTINCT transaction_id) dups, count_if(store_id <> upper(store_id)) lower FROM __SILVER__.fact_transactions")
check("Bronze has duplicate and lower-case defects", b.dups > 0 and b.lower > 0, f"{b.n:,} rows, {b.dups:,} duplicate IDs, {b.lower:,} lower-case IDs")
check("Silver defects cleaned", s.dups == 0 and s.lower == 0, f"{s.n:,} tickets, {s.dups} duplicates, {s.lower} lower-case")

rec = sql("""
  SELECT (SELECT sum(net_sales) FROM __SILVER__.fact_transactions WHERE NOT is_voided) silver,
         (SELECT sum(net_sales) FROM __GOLD__.store_daily_performance) store_day,
         (SELECT sum(net_sales) FROM __GOLD__.product_sales_daily) product_day,
         (SELECT sum(net_sales) FROM __GOLD__.store_hourly_traffic) store_hour""").first()
check("Net sales reconcile across layers", len({rec.silver, rec.store_day, rec.product_day, rec.store_hour}) == 1, f"${rec.silver:,.2f}")

t = one("""SELECT avg(CASE WHEN business_date >= DATE'2026-10-15' THEN transactions END) /
                  avg(CASE WHEN business_date BETWEEN DATE'2026-09-15' AND DATE'2026-10-14' THEN transactions END) - 1 AS chg
           FROM __GOLD__.store_daily_performance WHERE store_id = 'S007'""")
check("(a) S007 traffic drops from 2026-10-15", t.chg < -0.4, f"{t.chg:+.1%} vs prior 30 days")

inv = [r.k for r in sql("SELECT concat(store_id, ' ', item_name) k FROM __GOLD__.inventory_current_status WHERE stock_status = 'REORDER NOW' ORDER BY 1").collect()]
check("(b) Exactly 3 items to reorder", inv == ["S004 Oat Milk", "S008 Oat Milk", "S010 Espresso Beans"], ", ".join(inv))

p = sql("""SELECT CASE WHEN business_date >= DATE'2026-01-05' THEN 'after' ELSE 'before' END period, MEASURE(avg_ticket) t
                 FROM __GOLD__.store_sales_metrics WHERE is_comp_store AND business_date BETWEEN DATE'2025-10-05' AND DATE'2026-04-05' GROUP BY ALL""").collect()
p = {r.period: float(r.t) for r in p}
check("(c) Average ticket rises after the 2026-01-05 price increase", p["after"] > p["before"], f"${p['before']:.2f} -> ${p['after']:.2f}")

n = one("SELECT min(business_date) first_day, count_if(is_comp_store) comp FROM __GOLD__.store_daily_performance WHERE store_id = 'S012'")
check("(d) RiNo opened 2026-03-01, no comp days yet", str(n.first_day) == "2026-03-01" and n.comp == 0, f"first day {n.first_day}, {n.comp} comp days")

c = one("""SELECT avg(CASE WHEN month(business_date) BETWEEN 5 AND 8 THEN u END) / avg(CASE WHEN month(business_date) NOT BETWEEN 5 AND 8 THEN u END) ratio
           FROM (SELECT date_trunc('MONTH', business_date) business_date, sum(units_sold) u FROM __GOLD__.product_sales_daily
                 WHERE category = 'Cold Drinks' AND year(business_date) = 2025 GROUP BY 1)""")
check("(e) Cold drinks peak May-Aug", c.ratio > 3, f"{c.ratio:.1f}x the rest of 2025")

m = sorted({r.m for r in sql("SELECT DISTINCT month(business_date) m FROM __GOLD__.product_sales_daily WHERE product_name = 'Pumpkin Spice Latte'").collect()})
check("(f) Pumpkin Spice Latte only Sep-Nov", set(m) <= {9, 10, 11} and m, f"months {m}")

labor = [r.store_id for r in sql("""SELECT store_id FROM __GOLD__.store_daily_performance WHERE business_date BETWEEN DATE'2026-09-22' AND DATE'2026-10-19'
                                          GROUP BY ALL HAVING sum(labor_cost) / sum(net_sales) > 0.30""").collect()]
check("Labor alert has a store over 30%", len(labor) > 0, ", ".join(labor))

promo = one("SELECT count(*) n FROM __GOLD__.loyalty_promo_targets")
check("Promo target list built", promo.n > 0, f"{promo.n:,} members")

fp = one("""SELECT sha2(concat_ws('|',
    (SELECT array_join(sort_array(collect_list(concat_ws(',', store_id, business_date, transactions, net_sales, labor_cost))), ';') FROM __GOLD__.store_daily_performance),
    (SELECT array_join(sort_array(collect_list(concat_ws(',', member_id, visits, lifetime_net_sales, favorite_product, customer_segment))), ';') FROM __GOLD__.customer_360),
    (SELECT array_join(sort_array(collect_list(concat_ws(',', store_id, item_code, on_hand_qty, stock_status))), ';') FROM __GOLD__.inventory_current_status),
    (SELECT cast(count(*) AS STRING) FROM __BRONZE__.raw_pos_transactions)), 256) fp""").fp

displayHTML("<table style='font-family:sans-serif;font-size:13px;border-collapse:collapse;'>" + "".join(
    f"<tr><td style='padding:3px 10px;color:{'#2E7D32' if r == 'PASS' else '#C62828'};font-weight:700;'>{r}</td><td style='padding:3px 10px;'>{n}</td><td style='padding:3px 10px;color:#5D4037;'>{d}</td></tr>"
    for r, n, d in checks) + f"</table><p style='font-family:monospace;font-size:12px;color:#5D4037;'>Data fingerprint: {fp}<br/>Reference build: {EXPECTED_FINGERPRINT}</p>")
failed = [n for r, n, _ in checks if r == "FAIL"]
assert not failed, f"Validation failed: {failed}"
if fp != EXPECTED_FINGERPRINT:
    print("NOTE: the fingerprint differs from the reference build, so the build SQL above was probably edited.")

<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Step 6</div>
<div style="font-size:24px;font-weight:700;">&#128190; Deliverables: queries, data prep flow, dashboard, Genie agent, alerts</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Each cell below creates one kind of workspace asset, named <b>[dbsql_workshop &lt;you&gt;] &hellip;</b>, in the bundle's own folder (printed by the first cell). Re-running a cell updates its assets in place; the bundle's Destroy deletes the folder and everything in it. Alert and dashboard refresh schedules start paused, so the warehouse only runs when you use the demo.</p>

<div style="font-size:17px;font-weight:700;color:#5D4037;margin-top:6px;">&#128221; SQL editor script and saved queries 01&ndash;09</div>
<p style="color:#3E2723;font-size:14px;margin:4px 0;">One modern SQL feature per query: metric views, QUALIFY, <code>ai_query()</code> and more.</p>

In [ ]:
assets = WorkshopAssets(cfg)
assets.saved_queries()

<div style="font-size:17px;font-weight:700;color:#5D4037;margin-top:6px;">&#129513; Visual data prep flow</div>
<p style="color:#3E2723;font-size:14px;margin:4px 0;">Loyalty Promo Target List Prep: the no-code version of <code>gold.loyalty_promo_targets</code>.</p>

In [ ]:
assets.flow()

<div style="font-size:17px;font-weight:700;color:#5D4037;margin-top:6px;">&#128202; AI/BI dashboard</div>
<p style="color:#3E2723;font-size:14px;margin:4px 0;">Four pages on the gold tables and metric views, published with embedded credentials.</p>

In [ ]:
assets.dashboard()

<div style="font-size:17px;font-weight:700;color:#5D4037;margin-top:6px;">&#129302; Genie agent</div>
<p style="color:#3E2723;font-size:14px;margin:4px 0;">Instructions, worked examples, joins, starter questions and benchmarks on the gold tables and metric views.</p>

In [ ]:
assets.genie()

<div style="font-size:17px;font-weight:700;color:#5D4037;margin-top:6px;">&#128276; Alerts</div>
<p style="color:#3E2723;font-size:14px;margin:4px 0;">Inventory below reorder point, store traffic drop and labor cost above target. Weekly, Monday 8 AM New York, paused.</p>

In [ ]:
assets.alerts()

<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Step 7</div>
<div style="font-size:24px;font-weight:700;">&#128193; Asset index</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Live links to everything the deploy and this notebook created.</p>

In [ ]:
displayHTML(assets.index())